# 2.5 Python Dictionary Datatype

**Prerequisites:** 2.4 Python List-Array Datatype  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Key-value storage, and why lookup is O(1)
- Insertion order is guaranteed since Python 3.7
- Creating, accessing, updating and deleting pairs
- `get()` vs `[]` vs `setdefault()` — and when each is right
- Views (`keys`/`values`/`items`) and why they are live
- Merging with `|` and `|=` (3.9+)
- `defaultdict` and `Counter` for grouping and tallying

---

## Python Dictionary:
- A dictionary in Python is a collection of **key-value pairs**, written inside curly
  braces `{}` with `key: value` and separated by commas.
- Keys must be **unique** and **hashable** (immutable). Values can be anything.
- Dictionaries are **mutable** — you can add, change and remove pairs after creation.
- Lookup by key is **O(1)** on average, no matter how large the dictionary gets. This is
  the whole point of a dict.

> ### Version note — dictionaries are ordered now
> The original version of this note called dictionaries "unordered". That was true up to
> **Python 3.6**. Since **Python 3.7** *insertion order is preserved as a language
> guarantee*: iterating a dict yields keys in the order they were first added.
>
> You will still find "dicts are unordered" in a great many tutorials and StackOverflow
> answers. It is out of date.
>
> (`collections.OrderedDict` still exists, but you only need it now for its
> `move_to_end()` method and order-sensitive equality.)

**Analogy:** a list is a numbered row of lockers — you must know the number. A dictionary
is a coat check — you hand over a *ticket* (the key) and get your coat back, and it takes
the same time whether there are 10 coats or 10 million.

### Create empty dictionary:

In [ ]:
d0 = {}
d1 = dict()

In [ ]:
print(type(d0), len(d0))
print(type(d1), len(d1))

### Create non-empty dictionary:

In [ ]:
data1 = {'name':'aman', 'age':21, 'num':[50, 65, 67]}
data2 = dict({'name':'aman', 'age':21, 'num':[50, 65, 67]})
data3 = dict(name = 'aman', age = 21, num =[50, 65, 67])

In [ ]:
print(data1,len(data1))
print(data2,len(data2))
print(data3,len(data3))

In [ ]:
d = {}
r = int(input("Enter no. of pairs to store: "))
for i in range(r):
    k = input('Enter Key: ')
    v = input('Enter Value: ')
    d[k] = v
print(d)

### Access value of Dictionary:

In [ ]:
data1['age']

In [ ]:
# data1['day'] # Since 'day' key doesn't exist, this instruction will return error. 

- **get():** get() method returns:
    - the value for the specified key if key is in dictionary.
    - None if the key is not found and value is not specified.
    - value if the key is not found and value is specified.

In [ ]:
# if key exist
data1.get('age')

In [ ]:
# if the key doesn't exist and value is not specified.
print(data1.get('day'))

In [ ]:
# if the key doesn't exist and value is specified.
print(data1.get('day', 'Key not found'))

### Python Operators in Dictionary:
- **Assignment Operator(=):**

In [ ]:
# Update value of a key:
data1['name'] = 'vivek'
print(data1)

In [ ]:
# Add a key-value pair:
data1['sub'] = ['C', 'C++']
print(data1)

- **Membership Operator(in/not in):** 
    - Check for membership of a key or value.

In [ ]:
print(data1)
print('age' in data1.keys())
print(25 in data1.values())

> **`in` checks keys, not values.** `"name" in data` tests the *keys* — and does so in
> O(1). Writing `"name" in data.keys()` is equivalent but longer, so just write `in data`.
>
> Searching values (`21 in data.values()`) is a linear scan, O(n). If you need that often,
> you probably want a second dict mapping the other way.

In [ ]:
data = {"name": "aman", "age": 21}

# `in` on a dict checks KEYS by default - this is the fast, O(1) path
print("'name' in data        :", "name" in data)
print("'name' in data.keys() :", "name" in data.keys(), "  <- same thing, more typing")

# To search VALUES you must say so - and it is O(n), a linear scan
print("\n21 in data.values()   :", 21 in data.values())

# Why it matters: membership testing at scale
import time

big_dict = {i: i for i in range(200_000)}
big_list = list(range(200_000))
target = 199_999

start = time.perf_counter()
_ = target in big_dict
dict_time = time.perf_counter() - start

start = time.perf_counter()
_ = target in big_list
list_time = time.perf_counter() - start

print(f"\ndict lookup: {dict_time * 1_000_000:8.1f} microseconds")
print(f"list lookup: {list_time * 1_000_000:8.1f} microseconds")
print("This gap is the entire reason dictionaries exist.")

### Python Dictionary Methods

- **keys():** Return sequence of all keys of dictionary

In [ ]:
print(data1.keys())
print(list(data1.keys()))

- **values():** Return sequence of all values of dictionary

In [ ]:
print(data1.values())
print(list(data1.values()))

- **items():** Return a sequence of tuple inwhich each tuple is having key and value of dictionary

In [ ]:
print(data1.items())
print(list(data1.items()))

### Dictionary views are live

`.keys()`, `.values()` and `.items()` do **not** return lists. They return **view objects**
— live windows onto the dictionary. Change the dict and the view changes with it.

Two practical consequences:

1. **You cannot index a view.** `d.keys()[0]` is a `TypeError`. Wrap it: `list(d.keys())[0]`.
2. **Key views support set operations** — `d1.keys() & d2.keys()` gives you the common keys
   directly, which is genuinely useful and not widely known.

In [ ]:
d = {"a": 1, "b": 2, "c": 3}

keys = d.keys()
values = d.values()
items = d.items()

print("keys  :", keys)
print("type  :", type(keys))

# Views are LIVE - they reflect later changes to the dict
d["e"] = 5
print("\nafter adding 'e':")
print("  keys :", keys, "  <- updated automatically")

# Take a snapshot with list() when you need one
snapshot = list(d.keys())
d["f"] = 6
print("\nsnapshot :", snapshot, "  <- frozen")
print("live view:", keys)

# Key views support SET operations - very handy
other = {"a": 10, "b": 20, "z": 30}
print("\nkeys in both      :", d.keys() & other.keys())
print("keys only in d    :", d.keys() - other.keys())
print("keys in either    :", d.keys() | other.keys())

# ⚠️ Mutating while iterating raises
try:
    for k in d:
        if k == "a":
            del d[k]
except RuntimeError as exc:
    print("\nMutating during iteration:", exc)

# The fix: iterate over a snapshot
for k in list(d):
    if k == "a":
        del d[k]
print("Deleted safely   :", d)

### copy(): Create a copy of dictionary at different memory location.

In [ ]:
data2 = data1.copy() # Shallow copy
print(data2)

In [ ]:
data2['name'] = 'rohit'
print(data2)
print(data1)

### update(): Update Dictionary using Other Dictionary

In [ ]:
dict0 = {'name':'adi', 'age':21, 'roll':200}
dict1 = {'name':'priya', 'age':50, 'sub':['Jave', 'Python']}
print(dict0)
dict0.update(dict1)  # updating dict0
print(dict0)

### Merging dictionaries

> **Version note:** the `|` and `|=` operators for dicts were added in **Python 3.9** (PEP 584).

Four ways to combine dictionaries, in rough order of preference:

| Expression | Effect | Since |
|---|---|---|
| `a \| b` | **New** dict; `b` wins on conflicts | 3.9 |
| `a \|= b` | Updates `a` **in place** | 3.9 |
| `{**a, **b}` | **New** dict; `b` wins | 3.5 |
| `a.update(b)` | Updates `a` in place, returns `None` | always |

**Real-world use case:** layering configuration — defaults, then a config file, then
environment variables, then command-line flags, each overriding the last.

In [ ]:
defaults = {"host": "localhost", "port": 8080, "debug": False}
overrides = {"port": 9090, "debug": True}

# 1. | creates a NEW dict (3.9+). Right-hand side wins on conflicts.
merged = defaults | overrides
print("defaults |overrides:", merged)
print("defaults unchanged :", defaults)

# 2. |= updates IN PLACE (3.9+) - equivalent to .update()
config = defaults.copy()
config |= overrides
print("\nafter |=           :", config)

# 3. {**a, **b} - works on 3.5+, same result as |
print("{**a, **b}         :", {**defaults, **overrides})

# 4. .update() - mutates and returns None
old_style = defaults.copy()
result = old_style.update(overrides)
print("\n.update() returned :", result, " <- None, a common surprise")
print("old_style          :", old_style)

# Order matters - the LAST occurrence of a key wins
print("\noverrides | defaults:", overrides | defaults, " <- defaults win now")

# Merging several at once
extra = {"timeout": 30}
print("\nthree-way merge    :", defaults | overrides | extra)

### pop(): It will remove the pair and also return value.
- Syntax: object.pop(key)

In [ ]:
print(data1)
temp = data1.pop('name')
print(data1)
print(temp)

### popitem(): It will remove any arbitrary pair and also return the pair. 
- Syntax: object.popitem()
- It doesnot take any argument.

In [ ]:
print(data1)
temp = data1.popitem()
print(data1)
print(temp)

### clear(): Remove all items from dictionary

In [ ]:
print(data1)
data1.clear()
print(data1)

### del: 
- **Delete a particular key-value pair**

In [ ]:
print(data2)
del data2['age']
print(data2)

- **If value corresponding to a key is list:**

In [ ]:
print(data2)
del data2['sub'][1:2]   # start:stop of list
print(data2)

In [ ]:
print(data2)
del data2['num'][:]   # empty the value
print(data2)

In [ ]:
print(data2)
del data2  # delete entire dictionary
# print(data2)

### setdefault():
- It return the value of a key, if the key is in dictionary else it insert the key with a value in the dictionary (Default value is None).
- **Syntax:** object.setdefault(key,value=None)

In [ ]:
print(data3)
print(data3.setdefault('age'))
print(data3)

In [ ]:
print(data3)
print(data3.setdefault('perc'))
print(data3)

In [ ]:
print(data3)
print(data3.setdefault('board','CBSE'))
print(data3)

In [ ]:
print(data3)
# The original line here was:  data3.setdefault('num,'CBSE')
# - mismatched quotes, so this cell could never run. Corrected:
print(data3.setdefault('num', 'CBSE'))
print(data3)

# get() vs setdefault(): the crucial difference
sample = {'a': 1}
print("\nbefore        :", sample)
print("get('b', 99)  :", sample.get('b', 99))
print("after get     :", sample, "  <- unchanged")

print("setdefault('b', 99):", sample.setdefault('b', 99))
print("after setdefault   :", sample, "  <- 'b' was INSERTED")

---

### `defaultdict` and `Counter` — stop hand-rolling these

Two `collections` types solve the two things people most often write by hand with dicts.
They are covered fully in **07 Module and Packages**; meet them here because they belong
to the dictionary story.

| Problem | Hand-rolled | Better |
|---|---|---|
| Counting occurrences | `d[k] = d.get(k, 0) + 1` | `Counter` |
| Grouping into lists | `d.setdefault(k, []).append(v)` | `defaultdict(list)` |

In [ ]:
from collections import defaultdict, Counter

# ---- Counter: tallying ----
words = "the quick brown fox jumps over the lazy dog the fox".split()

# By hand
manual = {}
for w in words:
    manual[w] = manual.get(w, 0) + 1

# With Counter
counts = Counter(words)

print("manual == Counter:", manual == counts)
print("most common 3    :", counts.most_common(3))
print("count of 'fox'   :", counts["fox"])
print("missing key      :", counts["zebra"], "  <- returns 0, no KeyError")

# ---- defaultdict: grouping ----
people = [("Mumbai", "Aditya"), ("Delhi", "Priya"),
          ("Mumbai", "Rahul"), ("Pune", "Sneha"), ("Delhi", "Vikram")]

# By hand, with setdefault
manual_groups = {}
for city, name in people:
    manual_groups.setdefault(city, []).append(name)

# With defaultdict - the factory runs automatically on a missing key
groups = defaultdict(list)
for city, name in people:
    groups[city].append(name)

print("\nmanual == defaultdict:", manual_groups == groups)
for city, names in groups.items():
    print(f"  {city:<8} {names}")

# ⚠️ defaultdict CREATES the key on access - even a read
print("\nbefore reading 'Chennai':", list(groups))
_ = groups["Chennai"]
print("after  reading 'Chennai':", list(groups), "  <- it now exists!")

---

### Worked examples: signup and login

The three programs below build on each other — store a password, then validate it, then add
a login flow. They are kept as a deliberate progression rather than merged, so you can see
each requirement being layered on.

> ⚠️ **These store passwords in plain text.** That is fine for learning dictionary mechanics
> and completely unacceptable in real software. Real systems store a *salted hash*
> (`hashlib.scrypt` or the `bcrypt`/`argon2` libraries) and never the password itself.
> Security is covered properly in **20 Working with APIs**.

### WAP to store username and password in dictionary:
- if password length is greater than 5.

In [ ]:
data={}
print('----Sign up----')
name= input("Choose Username: ")
pwd= input("Choose Password: ")
if len(pwd)>=5:
    data[name]=pwd
    print('Account Created')
else:
    print('Read User Guideline')
print(data)

### WAP to store Username and Password in Dictionary 
- if username doesn't exist already.
- if password length is greater than 5.

In [ ]:
data={'aditya':123445}
print('----Sign up----')
name= input("Choose Username: ")
if name not in data.keys():
    pwd= input("Choose Password: ")
    if len(pwd)>=5:
        data[name]=pwd
        print('Account Created')
    else:
        print('Read User Guideline')
else:
    print('Try Using different Username')
        
print(data)

### WAP for Signup and Login:

In [ ]:
data={'aditya':123445}
print('Choose Option:\n1. Signup \n2.Login')
ch=int(input("Enter choice no.: "))
if ch==1:
    print('----Sign up----')
    name= input("Choose Username: ")
    if name not in data.keys():
        pwd= input("Choose Password: ")
        if len(pwd)>=5:
            data[name]=pwd
            print('Account Created')
        else:
            print('Read User Guidelines.')
    else:
        print('Try Using different Username')
elif ch==2:
    print("-------Login-------")
    name = input("Enter username: ")
    if name in data.keys():
        pwd = input("Enter password: ")
        if pwd==data[name]:
            print("Welcome, Login Successful")
        else:
            print("Wrong Password")
    else:
        print("Unknown User")
else:
    print('Unknown Choice')

---

## Common Mistakes & Pitfalls

1. **Calling dicts unordered.** Since **Python 3.7** insertion order is a language guarantee. Older tutorials (and the original version of this notebook) say otherwise.
2. **Using `d[key]` when the key might be missing.** Raises `KeyError`. Use `d.get(key)`, `d.get(key, default)`, or `key in d` first.
3. **Confusing `d.get(k, default)` with `d.setdefault(k, default)`.** `get()` only reads; `setdefault()` **also inserts** the default when the key is absent.
4. **Using a mutable object as a key.** Keys must be hashable — `list` and `set` cannot be keys, `tuple` and `frozenset` can.
5. **Mutating a dict while iterating it.** `RuntimeError: dictionary changed size during iteration`. Iterate over `list(d)` or `list(d.items())` if you must modify.
6. **Assuming `.keys()` returns a list.** It returns a *view* — a live window onto the dict. Wrap it in `list()` if you need a snapshot.
7. **Expecting `d1.update(d2)` to return the merged dict.** It mutates `d1` and returns `None`. Use `d1 | d2` (3.9+) if you want a new dict.
8. **Using `dict.fromkeys(keys, [])`.** Every key gets *the same* list object.

## Best Practices

- Use `.get()` with a default for optional lookups; let `d[key]` raise when absence is a bug.
- Use `collections.defaultdict` for grouping and `collections.Counter` for tallying — don't hand-roll them.
- Use `|` / `|=` (3.9+) to merge dicts; `{**a, **b}` for older versions.
- Iterate with `.items()` when you need both key and value.
- Use dict comprehensions to build and transform dicts (see **03 Flow Control**).
- Use tuples as composite keys — `sales[('Mumbai', 2024)]` is often better than nesting.
- Prefer a `dataclass` or `NamedTuple` over a dict when the keys are fixed and known.

## Practice Exercises

Try these before moving on.

1. Count word frequency in a sentence — first with a plain dict, then with `Counter`.
2. Group a list of `(city, name)` pairs into `{city: [names]}` using `defaultdict`.
3. Invert a dictionary so values become keys. What happens when two keys share a value?
4. Merge two config dicts so the second wins on conflicts, using three different methods.
5. Given `{'a': 3, 'b': 1, 'c': 2}`, produce a new dict sorted by value.
6. Why can a tuple be a dict key but not a list? Demonstrate both.
7. Write a nested lookup `get_nested(data, 'user', 'address', 'city')` that returns `None` instead of raising if any level is missing.
8. Show the `RuntimeError` from deleting keys while iterating, then fix it two ways.